### Guardrails with Langchain:
 - What are Guardrails & Why do they matter?
 - Two approaches: Deterministic vs Model-based
 -  Built-in: PII Detection Middleware
 - Built-in: Human-in-the-Loop Middleware
 - Custom: Before-Agent Guardrail (input filtering)
 - Custom: After-Agent Guardrail (output safety)
 - Layered / Combined Guardrails
 -  Real-World Use Case: Healthcare Chatbot

In [1]:
#Installation:
from dotenv import load_dotenv
load_dotenv()

#Setup API keys:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

### What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

 - Before the agent starts (input guardrails)
 - After it completes (output guardrails)
 - Around model and tool calls





# Two Approaches to Guardrails:

### Deterministic Guardrails:
 - Rule-based: regex, keyword matching, explicit checks
 - ✅ Fast, predictable, cost-effective
 - ❌ May miss nuanced violations

### Model-Based Guardrails:
 - Uses LLMs/classifiers for semantic understanding
 - ✅ Catches subtle/nuanced issues
 - ❌ Slower and more expensive

## Common Use Cases

| Use Case | Example |
|----------|---------|
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |

In [8]:
### Quick Illustration of two approaches to Guardrails:
import re

# 1. Deterministic Guardrails: These are rules that are hard-coded into the system. They are designed to prevent the AI from taking certain actions or making certain decisions. For example, a deterministic guardrail might prevent the AI from accessing certain sensitive data or from making decisions that could harm users.

def deterministic_guardrail(input_text:str)-> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit","malware", "bomb", "attack"]
    return any(kw in input_text.lower() for kw in banned_keywords)

test_inputs = [
    "How to hack into a computer system?",
    "What are the best practices for cybersecurity?",
    "explain how to create malware",
]
print("=== Deterministic Guardrail Results ===")
for inp in test_inputs:
    if deterministic_guardrail(inp):
        print(f"Blocked: {inp}")
    else:
        print(f"Allowed: {inp}")

=== Deterministic Guardrail Results ===
Blocked: How to hack into a computer system?
Allowed: What are the best practices for cybersecurity?
Blocked: explain how to create malware


In [ ]:
#Model based approach:
from langchain_openai import ChatOpenAI

def model_based_guardrail(input_text:str)-> str:
    """use a LLM to evaluate content safefty. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
    prompt = f""" Is the following user input safe or unsafe? Please respond with only 'SAFE' or 'UNSAFE'.

    User Input: {input_text}

    """
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()

print("=== Model-based Guardrail Results ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if verdict == "UNSAFE" else "SAFE"
    print(f"Input: {inp} | Verdict: {status}")

 

### Built-in Guardrail — PII Detection Middleware:
 - LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

#Define a simple dummy tool:
def customer_lookup(query:str)-> str:
    """Dummy tool to simulate a customer lookup."""
    return f"Customer info for query: {query}"

#create agent with PII middleware:
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [customer_lookup],
    middleware = [
        # Redact emails in user input before sending to the model
        PIIMiddleware(
            "email",  # Type of PII to redact
            strategy="redact",  # Redaction strategy
            apply_to_input=True,  # Apply to user input
        ),
        #Mask credit cards in user input before sending to the model
        PIIMiddleware(
            "credit_card",  # Type of PII to mask
            strategy="mask",  # Masking strategy
            apply_to_input=True,  # Apply to user input
        ),

        #block API keys in user input before sending to the model---raise error if detected
        PIIMiddleware(
            "api_key",  # Type of PII to block
            detector = r"sk-[A-Za-z0-9]{48}",  # Regex pattern to detect API keys
            strategy="block",  # Blocking strategy
            apply_to_input=True,  # Apply to user input
        ),
    ],
)

print("=== Agent with PII Middleware created successfully ===")

In [ ]:
#Test PII Redaction:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is test@example.com and my credit card number is 4111 1111 1111 1111. Can you help?"
    }]
})

print("Agent Response after PII Redaction:")
print(result["messages"][-1].content)


In [ ]:
result

In [ ]:
#Test API Key Blocking:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "My OpenAI API key is sk-1234567890abcdef1234567890abcdef1234567890abcdef. Can you help?"
        }]
    })
except Exception as e:
    print("Agent Response after API Key Blocking:")
    print(e)

### Built-in Guardrail — Human-in-the-Loop Middleware:
 - Pauses agent execution before sensitive operations and waits for human approval.

## Best for:
 - Financial transactions
 - Sending emails to external parties
 - Deleting production data
 - Any operation with significant business impact
 
Key requirement: A checkpointer for state persistence across interrupts.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for query: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to the specified recipient."""
    return f"Email sent to {to} with subject '{subject}' and body '{body}'"

@tool
def delete_record(table: str, condition: str) -> str:
    """Delete a record from the database."""
    return f"Deleted records from  {table} where {condition}"

#Create an agent with Human-in-the-Loop middleware:
hitl_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_web, send_email, delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # Require human approval for sending emails
                "delete_record": True,  # Require human approval for deleting records
                "search_web": False,  # No human approval needed for web searches
            }
        )
    ],
    checkpointer = InMemorySaver() #required for state persistence across human approvals
)
print("=== Agent with Human-in-the-Loop Middleware created successfully ===")

In [ ]:
# Step 1: Invoke — agent will pause before send_email:
config = {"configurable":{"thread_id":"session_001"}}

result = hitl_agent.invoke({
    "messages": [{"role": "user", "content": "Please send an email to shamser@company.com about quarterly report."}],
    "config": config
})

print("=== Agent paused --- awaiting human approval ===")
print(result["messages"][-1].content)  # This will show the agent's response indicating it's awaiting approval

In [ ]:
#step 2 : Human review and approval:
# Simulate human approval by sending a follow-up message to the agent
approved_result = hitl_agent.invoke({
    Command(resume={"decision": [{"type": "approve"}]})"}),
    config = config #same thread_id to continue the same session
)

print("=== Approved! final response ===")
print(approved_result["messages"][-1].content)  # This will show the agent's final response after approval

In [ ]:
#step 3 : Human review and rejection:
config2 = {"configurable":{"thread_id":"session_002"}}

hitl_agent.invoke(
    "messages": [{"role": "user", "content": "Please delete the record from the users table where active= false."}],
    "config": config2
)


rejected_result = hitl_agent.invoke(
    Command(resume={"decision": [{"type": "reject", "reason": "This action is not allowed."}]}),
    config = config2 #same thread_id to continue the same session
)

print("=== Rejected! final response ===")
print(rejected_result["messages"][-1].content)  # This will show the agent

### Custom Guardrail — Before-Agent Hook (Input Filter):

Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

   - Keyword/content filtering
   - Authentication checks
   - Rate limiting
   -  Blocking specific categories of requests_

In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Determinitic guardrail middleware that checks for banned keywords in user input and raises an error if any are found.
    This runs before the agent processes the input, ensuring that any content containing banned keywords is blocked from being processed further.
    zero llm cost, deterministic, and fast.

    """
    
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]
    
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked --banned keyword detected: '{keyword}'")
                return{
                    "messages": [{
                        "role": "assistant", 
                        "content": (
                            "I can not process this request as it contains inappropiate content."
                            "Please rephrase your request without using any banned keywords."
                        )
                    }],
                    "jump_to": "end"

                }
            
            return None


@tool
def search_tool(query: str) -> str:
    """A simple search tool for information."""
    return f"Search results for query: {query}"


#create agent with Content filter:
filtered_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        )
    ],
)
print("=== Agent with Content Filter Middleware created successfully ===")


In [ ]:
#Test 1 : Safe request should pass through.
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "what is attention mechanism in transformers?"}]
})  
print("=== Safe Request Result ===")
print(result["messages"][-1].content)  # This will show the agent's response

In [ ]:
#Test 2: Unsafe request should be blocked.
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How to hack into a computer system?"}]
})
print("=== Unsafe Request Result ===")
print(result["messages"][-1].content)  # This will show the agent's response

### Custom Guardrail — After-Agent Hook (Output Safety):

Use after_agent() to validate the final agent response before the user sees it.

## Best for:
 - Model-based safety evaluation of outputs
 - Compliance scanning (e.g. legal, medical, financial disclaimers)
 - Quality validation
 - Removing sensitive info that slipped through

In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from sympy import content
from typer import prompt

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail middleware that uses a LLM to evaluate the safety of user input.
    If the model determines the input is unsafe, it raises an error and prevents further processing.
    This approach allows for more nuanced safety checks but incurs LLM costs and may be slower than deterministic checks.
    """
    
    def __init__(self, model_name: str = "gpt-4o-mini"):
        super().__init__()

        #use a smaller, cheaper model for safety checks to reduce costs
        self.safety_model = ChatOpenAI(model_name=model_name, temperature=0)
    
    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use the light weight model to evaluate the safety of the content
        safety_prompt = f"""Is the following response safe or unsafe? Please respond with only 'SAFE' or 'UNSAFE'.
response to evaluate: "{last_message.content}"
"""
        response = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in response.content.upper():
            print("Output blocked -- model determined content is unsafe.")
            last_message.content = (
                "I can not process this request as it contains inappropiate content."
                "Please rephrase your request without using any unsafe content."
            )
        return None


@tool
def general_tool(query: str) -> str:
    """A simple tool for general information."""
    return f"General tool results for query: {query}"

safe_agent = create_agent(
    model="gpt-4o-mini",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)
print("=== Agent with Safety Guardrail Middleware created successfully ===")

In [ ]:
#Test output safety check:
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "Please provide instructions on how to hack into a computer system."}]
})
print("=== Safety Guardrail Test Result ===")
print(result["messages"][-1].content)  # This will show the agent's response

### Layered / Combined Guardrails:
Stack multiple guardrails in the `middleware=[]` array. They execute in order, building layered protection.

```
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response
```

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Production-grade agent with 5-layer guardrails created!")

### Real-World Use Case — Healthcare Chatbot:
A healthcare chatbot that:
 - Blocks off-topic or harmful requests
 - Redacts patient PII (emails, credit card numbers)
 - Requires human approval before booking appointments
 - Validates that outputs are medically appropriate

In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    
# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."

# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model="gpt-4o",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

In [ ]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result

In [ ]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

In [ ]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

In [ ]:
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)

## Summary

| Guardrail Type | Hook | When it Runs | Best For |
|---|---|---|---|
| PII Middleware | Input/Output | Around model calls | Data privacy, compliance |
| Human-in-the-Loop | Tool level | Before sensitive tools | High-stakes decisions |
| Content Filter | `before_agent` | Start of invocation | Blocking bad inputs early |
| Safety Validator | `after_agent` | End of invocation | Output quality/safety |
| Custom Logic | Any hook | Anywhere | Any business rule |

### 🔑 Key Takeaways

1. **Guardrails = Middleware** — implement them via the `middleware=[]` parameter in `create_agent()`
2. **Layer your guardrails** — defense in depth is best practice
3. **Deterministic first, model-based second** — use cheap rule-based checks early to avoid expensive LLM calls
4. **Human-in-the-Loop requires a checkpointer** — use `InMemorySaver` for dev, persistent store for production
5. **Custom middleware** gives you full control via `before_agent()` and `after_agent()` hooks